## About the dataset

A test data set of just 20 rows, downloaded from kaggle to practice.

**STEP 1: Data Ingestion**<br>
properly read data via the CSV file.

In [33]:
import pandas as pd
import os
import numpy as np

# file path 
file_path: str = '../messy_data_sets/Wiki_dataset.csv'

# Check wether the file exists and then open it.
if os.path.exists(file_path):
    try:
        df: pd.DataFrame = pd.read_csv(
            file_path,
            on_bad_lines='warn', # For any courrupted, line or rows.
            # Null value Standardization,
            na_values= [' ', '', 'Missing', 'missing', 'Unknown', 'unknown', 'N/A', 'NaN'],
            engine='c', # Performance
        )
        print(f'Data Ingestion succesfull!\ncontains {len(df.columns)} columns and {len(df)} rows')
    except Exception as e:
        print(f"Data Ingestion Failed {e}")
else:
    print(f'{file_path} file not exists!')

Data Ingestion succesfull!
contains 11 columns and 20 rows


**STEP 2: Data Profiling and Inspection**<br>
Using some common Pandas inspection tool to get a quick review of dataset.
such as:
`df.info()`
`df.describe()`
`df.shape`
`df.null.sum()`

In [34]:
# To know the shape of the dataset.
df.shape
print(f'The dataset contains a total of {df.shape[0]} rows and {df.shape[1]} columns.')

The dataset contains a total of 20 rows and 11 columns.


In [35]:
# To get a quick check of pandas preassigned value.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   Rank                              20 non-null     int64
 1   Peak                              9 non-null      str  
 2   All Time Peak                     6 non-null      str  
 3   Actual gross                      20 non-null     str  
 4   Adjusted gross (in 2022 dollars)  20 non-null     str  
 5   Artist                            20 non-null     str  
 6   Tour title                        20 non-null     str  
 7   Year(s)                           20 non-null     str  
 8   Shows                             20 non-null     int64
 9   Average gross                     20 non-null     str  
 10  Ref.                              20 non-null     str  
dtypes: int64(2), str(9)
memory usage: 3.4 KB


In [36]:
# To get a quick statistical summary of dataset's numericals columns.
df.describe()

,Rank,Shows
count,20.000000,20.000000
mean,10.450000,110.000000
std,5.942488,66.507617
min,1.000000,41.000000
25%,5.750000,59.000000
50%,10.500000,87.000000
75%,15.250000,134.500000
max,20.000000,325.000000


In [37]:
# let's see the total count of null values.
df.isnull().sum() 
# We have null values in peak and All Time peak.

Rank                                 0
Peak                                11
All Time Peak                       14
Actual gross                         0
Adjusted gross (in 2022 dollars)     0
Artist                               0
Tour title                           0
Year(s)                              0
Shows                                0
Average gross                        0
Ref.                                 0
dtype: int64

**STEP 3: Structural Standardization**<br>
handling empty white space around the columns names, compeletely empty columns used for human readability, Normalizing the column in a single concept, either lower or upper or snake format ('hell_world').


In [38]:
df.columns = df.columns.str.strip().str.lower()

columns_name: list = ['rank', 'peak', 'all time best', 'actual gross',
                      'adjusted gross', 'artist', 'tour title', 'years',
                      'shows', 'average gross', 'ref.']

# Instead of filtering each word inside columns , I just the name of entire data
# set columns.
df.columns = columns_name
# Replacing the space between the word with '_' to make it snake formated.
df.columns = df.columns.str.replace({' ' : '_', '.': ''}).str.strip()

# Remove completely empty columns used for human readability.
df = df.dropna(axis=1, how='all')

**STEP 4: Deduplication**<br>
Check for duplicate and handle accordingly it's type:
- Exact duplicate
- Partial duplicate

In [ ]:
# let's check for exact duplicates.
df.duplicated().sum() # output is 0 i.e. there's not any exact duplicate in dataset.

# let's check for partial duplicates.
# We'll do it by checkin each columns, if they contains any duplicate or not.
spaces = 15
for column in df.columns:
    print(f'{column}{' '*(spaces-len(column))}{df[column].duplicated().sum()}')

# let's start with the top (The 'rank' column)
# I'm seeing it has 1 duplicate, which shouldn't be, let's handle it via the 
# help of actual_gross , the ranking is defined in terms of who earned more,
# I have use the business logic here. now just make it 8. we will use the loc
# method for it, to target specifically that particular value in rank.

mask = (df['artist']=='Pink') & (df['tour_title'].str.contains('Summer Carnival'))
df.loc[mask, 'rank'] = 8 # solved.

rank           0
peak           12
all_time_best  13
actual_gross   0
adjusted_gross 0
artist         11
tour_title     0
years          4
shows          2
average_gross  0
ref            0


**Befor we go to datatype casting, first clean the numerical values**<br>

if we look at our dataset the columms 'actual_gorss', 'acjusted_gross', and 'average_gross', are the columns which contains currency value, let's clean it strip the currency symbols , and if we look closely we might encounter some extra tag e.g. [e] along with it. let's handle it via regex.

In [ ]:
# was for the testing purpose.
df['actual_gross'] = df['actual_gross'].str.replace(r'[^\d]', '', regex=True)
# I did with multiple columns at the same time, let's add the ref column too.
df[['adjusted_gross', 'average_gross', 'ref']] = df[['adjusted_gross', 'average_gross', 'ref']].replace(r'[^\d]', '', regex=True)

# let's do the same for peak and all_time_best columns.
df[['peak', 'all_time_best']]=df[['peak', 'all_time_best']].replace(r'[^\d]', '', regex=True)
df

,rank,peak,all_time_best,actual_gross,adjusted_gross,artist,tour_title,years,shows,average_gross,ref
0,1,1,2,780000000,780000000,Taylor Swift,The Eras Tour †,2023–2024,56,13928571,1
1,2,1,72,579800000,579800000,Beyoncé,Renaissance World Tour,2023,56,10353571,3
2,3,14,25,411000000,560622615,Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,4835294,6
3,4,27,107,397300000,454751555,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795,7
4,5,24,NaN,345675146,402844849,Taylor Swift,Reputation Stadium Tour,2018,53,6522173,8
5,6,24,109,305158363,388978496,Madonna,The MDNA Tour,2012,88,3467709,9
6,7,210,NaN,280000000,381932682,Celine Dion,Taking Chances World Tour,2008–2009,131,2137405,11
7,8,NaN,NaN,257600000,257600000,Pink,Summer Carnival †,2023–2024,41,6282927,12
8,9,NaN,NaN,256084556,312258401,Beyoncé,The Formation World Tour,2016,49,5226215,13
9,10,NaN,NaN,250400000,309141878,Taylor Swift,The 1989 World Tour,2015,85,2945882,14
